# Training

Trains one or more training profiles, one after another. Set `PROFILES` and run every cell. Profiles are
in `src/erosion_sim/config.py`; each plan step names the ones it needs. A world version is two profiles,
its coarse and fine level.

The best checkpoint (lowest validation loss) is saved to `artifacts/checkpoints/<OBJECTIVE>-<profile>.pt`,
and the latest state to `<OBJECTIVE>-<profile>-last.pt` at every validation. Set `RESUME = True` to
continue interrupted runs from those. `OVERRIDES` replaces config fields for every profile, for example
`{"steps": 5_000}`; leave it empty for the profile as written.

The data must exist first: `data.ipynb` with each profile's `data_profile`.

In [ ]:
from dataclasses import asdict

from erosion_sim.config import get_config
from erosion_sim.paths import DATA_DIR
from erosion_sim.train import train

PROFILES = []  # for example ["world_coarse", "world_fine"]
OBJECTIVE = "diffusion"
OVERRIDES = {}
RESUME = False

configs = [get_config(name, OBJECTIVE) for name in PROFILES]
for cfg in configs:
    missing = [s for s in ("train", "val") if not (DATA_DIR / cfg.data_profile / s / "meta.json").exists()]
    if missing:
        raise FileNotFoundError(f"{cfg.profile}: no {cfg.data_profile} {', '.join(missing)} data; run data.ipynb")
    print(f"=== {cfg.profile}")
    for key, value in {**asdict(cfg), **OVERRIDES}.items():
        print(f"{key:<18} {value}")

## Train

In [ ]:
results = {}
for cfg in configs:
    print(f"=== {cfg.profile}")
    history = train(cfg, resume=RESUME, **OVERRIDES)
    results[cfg.profile] = history
    print(f"best checkpoint: {history['checkpoint']}  ({history['seconds'] / 60:.1f} min)")